In [14]:
JSON_PATH    = "C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Data\\Neural Data\\DBS ATN DSB Case 1\\D. Siragusa\\3.5.26\\Time stamp 1426\\Report_Json_Session_Report_20260305T151332.json"
EPRIME_CSV   = "C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Data\\Eprime Data\\Digit Span Backwards v3.2\\DigitSpanBackward v3.3-6-1-Scores.csv"
EVENTS_PRE   = "C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Preprocessed Data\\events_data_processed.csv"       # from Script 2
OUTPUT_DIR   = "C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Preprocessed Data"


In [18]:
"""
=============================================================================
PIPELINE: E-Prime Preprocessor  +  Events Data Preprocessing
=============================================================================
INPUT FILES
  EPRIME_CSV   : DigitSpanBackward raw E-Prime export (.csv, skiprows=1)
  JSON_PATH    : Medtronic Percept session JSON
OUTPUT FILES
  EVENTS_PRE   : intermediate per-sub-trial table (in memory only)
  OUTPUT_PATH  : Events.csv  — final event marker stream

TASK STRUCTURE
  7 blocks, 14 trials, 52 sub-trials total
  Block 1:    2 trials × span 2  =  4 sub-trials
  Block 2:    2 trials × span 3  =  6 sub-trials
  Blocks 3-6: 2 trials × span 4  = 32 sub-trials
  Block 7:    2 trials × span 5  = 10 sub-trials

TIMING COLUMNS USED
  Welcome.TargetOnsetTime          → Session Start        (code  6)
  Goodbye.OffsetTime               → Session End          (code  7)
  Fixation.TargetOnsetTime         → Fixation Start       (code 16)
                                     Main Trial Start     (code  8)
  Fixation.OffsetTime              → Fixation End         (code 17)
  Stimulus.TargetOnsetTime         → Stimulus Start       (code 10)
  Stimulus.OffsetTime              → Stimulus End         (code 11)
  CollectResponse.TargetOnsetTime  → Choice Start         (code 12)
  CollectResponse.OffsetTime       → Choice End           (code 13)
  Feedback.TargetOnsetTime         → Feedback Start       (code 14)
  Feedback.OffsetTime              → Feedback End         (code 15)
                                     Main Trial End       (code  9)

EVENT CODE MAPPING
   6  Session Start
   7  Session End
   8  Main Trial Start      (= first Fixation.TargetOnsetTime of trial)
   9  Main Trial End        (= last Feedback.OffsetTime of trial)
  10  Stimulus Start
  11  Stimulus End
  12  Choice Start
  13  Choice End
  14  Feedback Start
  15  Feedback End
  16  Fixation Start
  17  Fixation End
  18  Stimulation Spike     (0 → 0.1 mA onset)
  19  Stimulation Spike End (0.1 → 0 mA offset)

OUTPUT COLUMNS (15)
  Time_ms          E-Prime clock time (ms, zero = 19:27:06 UTC)
  Time_s           same in seconds
  Event_Code       integer 6–19
  Event_Type       human-readable label
  Trial_Number     1-based, NaN outside trials
  Block            block number
  Span_Size        digits in this trial
  Sub_Trial        digit position (per-digit events only)
  Digit            digit shown on screen (per-digit events only)
  Window           Stimulus | Choice | Feedback | NaN
  Time_Relative_s  seconds from neural recording start (negative = pre-recording)
  Sample_Index     index into 250 Hz neural array (negative = pre-recording)
  ACC              1=correct, 0=incorrect
  CRESP            correct response
  RESP             patient response
=============================================================================
"""

import numpy as np
import pandas as pd
import json
import os
from datetime import datetime, timezone

# ── CONFIG ────────────────────────────────────────────────────────────────────
JSON_PATH    = "C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Data\\Neural Data\\DBS ATN DSB Case 1\\D. Siragusa\\3.5.26\\Time stamp 1426\\Report_Json_Session_Report_20260305T151332.json"
EPRIME_CSV   = "C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Data\\Eprime Data\\Digit Span Backwards v3.2\\DigitSpanBackward v3.3-6-1-Scores.csv"
OUTPUT_DIR  = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "Events.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPRIME_ZERO_UTC = datetime(2026, 3, 5, 19, 27, 6, tzinfo=timezone.utc)
NEURAL_START_MS = 348_000   # E-Prime ms when neural recording began

# ── EVENT CODE MAPPING ────────────────────────────────────────────────────────
EVENT_MAPPING = {
    6:  'Session Start',
    7:  'Session End',
    8:  'Main Trial Start',
    9:  'Main Trial End',
    10: 'Stimulus Start',
    11: 'Stimulus End',
    12: 'Choice Start',
    13: 'Choice End',
    14: 'Feedback Start',
    15: 'Feedback End',
    16: 'Fixation Start',
    17: 'Fixation End',
    18: 'Stimulation Spike',
    19: 'Stimulation Spike End',
}

print("=" * 70)
print("PIPELINE: E-Prime Preprocessor + Events Data Preprocessing")
print("=" * 70)

# ══════════════════════════════════════════════════════════════════════════════
# PART 1 — Build EVENTS_PRE (per-sub-trial table from raw E-Prime CSV)
# ══════════════════════════════════════════════════════════════════════════════
print("\n[PART 1] Building per-sub-trial table from E-Prime CSV...")

df_raw       = pd.read_csv(EPRIME_CSV, header=None)
column_names = df_raw.iloc[1].tolist()
df           = pd.DataFrame(df_raw.values[2:], columns=column_names)
print(f"    Raw shape : {df.shape}")

relevant_cols = {
    # Identity
    "SessionDate"                    : "session_date",
    "Block"                          : "block",
    "Trial"                          : "trial",
    "SubTrial"                       : "sub_trial",
    "CurrentSpanSize[Trial]"         : "span_size",
    "Digit"                          : "digit_shown",

    # Timing — TargetOnsetTime used for onset markers (scheduled presentation)
    #          OffsetTime used for offset markers (actual disappearance)
    "Welcome.TargetOnsetTime"        : "session_start_ms",
    "Goodbye.OffsetTime"             : "session_end_ms",
    "Fixation.TargetOnsetTime"       : "fixation_onset_ms",
    "Fixation.OffsetTime"            : "fixation_offset_ms",
    "Stimulus.TargetOnsetTime"       : "stimulus_onset_ms",
    "Stimulus.OffsetTime"            : "stimulus_offset_ms",
    "CollectResponse.TargetOnsetTime": "response_onset_ms",
    "CollectResponse.OffsetTime"     : "response_offset_ms",
    "Feedback.TargetOnsetTime"       : "feedback_onset_ms",
    "Feedback.OffsetTime"            : "feedback_offset_ms",

    # Response & accuracy
    "CollectResponse.RESP"           : "response_made",
    "CollectResponse.ACC"            : "is_correct",
    "CollectResponse.RT"             : "reaction_time_ms",
    "CorrectResp"                    : "correct_response",

    # Extra timing
    "Stimulus.Duration"              : "stimulus_duration_ms",
    "Stimulus.OnsetDelay"            : "stimulus_onset_delay_ms",
    "CollectResponse.Duration"       : "response_window_ms",
    "Feedback.Duration"              : "feedback_duration_ms",
}

existing = {k: v for k, v in relevant_cols.items() if k in df.columns}
missing  = [k for k in relevant_cols if k not in df.columns]
if missing:
    print(f"    ⚠ Columns not found (skipped): {missing}")

ep = df[list(existing.keys())].copy().rename(columns=existing)

# Numeric casting
numeric_cols = [
    "block", "trial", "sub_trial", "span_size", "digit_shown",
    "session_start_ms", "session_end_ms",
    "fixation_onset_ms", "fixation_offset_ms",
    "stimulus_onset_ms", "stimulus_offset_ms",
    "response_onset_ms", "response_offset_ms",
    "feedback_onset_ms", "feedback_offset_ms",
    "reaction_time_ms", "is_correct",
    "stimulus_duration_ms", "stimulus_onset_delay_ms",
    "response_window_ms", "feedback_duration_ms",
]
for col in numeric_cols:
    if col in ep.columns:
        ep[col] = pd.to_numeric(ep[col], errors="coerce")

ep = ep.dropna(subset=["fixation_onset_ms"]).reset_index(drop=True)
ep = ep.sort_values(["block", "trial", "sub_trial"]).reset_index(drop=True)

# Derived columns
ep["fixation_duration_ms"]       = ep["fixation_offset_ms"] - ep["fixation_onset_ms"]
ep["blank_screen_ms"]            = ep["stimulus_onset_ms"]  - ep["fixation_offset_ms"]
ep["stimulus_actual_duration_ms"]= ep["stimulus_offset_ms"] - ep["stimulus_onset_ms"]
ep["isi_to_next_digit_ms"]       = (
    ep.groupby(["block","trial"])["fixation_onset_ms"]
    .transform(lambda x: x.shift(-1) - x)
)
last_stim_offset = ep.groupby(["block","trial"])["stimulus_offset_ms"].transform("max")
ep["encoding_to_recall_gap_ms"]  = ep["response_onset_ms"] - last_stim_offset
ep["neural_data_available"]      = (ep["fixation_onset_ms"] >= NEURAL_START_MS).astype(int)

print(f"    Sub-trials loaded : {len(ep)}")
print(f"    Blocks            : {ep['block'].nunique()}")
print(f"    Session start     : {ep['session_start_ms'].iloc[0]:.0f} ms  (Welcome.TargetOnsetTime)")
print(f"    Session end       : {ep['session_end_ms'].iloc[0]:.0f} ms   (Goodbye.OffsetTime)")

# ══════════════════════════════════════════════════════════════════════════════
# PART 2 — Load Neural JSON
# ══════════════════════════════════════════════════════════════════════════════
print("\n[PART 2] Loading neural JSON...")

with open(JSON_PATH) as f:
    nd = json.load(f)

td        = nd["BrainSenseTimeDomain"][0]
SR        = float(td["SampleRateInHz"])
N_SAMPLES = len(td["TimeDomainData"])
first_pkt = datetime.fromisoformat(td["FirstPacketDateTime"].replace("Z", "+00:00"))
NEURAL_START_MS = (first_pkt - EPRIME_ZERO_UTC).total_seconds() * 1000.0
NEURAL_END_MS   = NEURAL_START_MS + (N_SAMPLES - 1) / SR * 1000.0

print(f"    Neural SR       : {SR} Hz")
print(f"    Neural samples  : {N_SAMPLES:,}")
print(f"    Neural window   : {NEURAL_START_MS:.0f} – {NEURAL_END_MS:.0f} ms "
      f"({(NEURAL_END_MS - NEURAL_START_MS)/1000:.2f} s)")

# LFP / spike detection
lfp_raw     = nd["BrainSenseLfp"][0]["LfpData"]
lfp_pkt_utc = datetime.fromisoformat(
                  nd["BrainSenseLfp"][0]["FirstPacketDateTime"].replace("Z", "+00:00"))
lfp_off_ms  = (lfp_pkt_utc - EPRIME_ZERO_UTC).total_seconds() * 1000.0
first_tick  = lfp_raw[0]["TicksInMs"]

lfp_rows = []
for e in lfp_raw:
    ep_ms = (e["TicksInMs"] - first_tick) + lfp_off_ms
    lfp_rows.append({
        "eprime_ms" : ep_ms,
        "left_ma"   : e["Left"]["mA"],
        "right_ma"  : e["Right"]["mA"],
        "left_lfp"  : e["Left"]["LFP"],
        "right_lfp" : e["Right"]["LFP"],
    })
lfp_df = pd.DataFrame(lfp_rows)
lfp_df["prev_left"] = lfp_df["left_ma"].shift(1).fillna(0)

spike_on_df  = lfp_df[(lfp_df["prev_left"] == 0) & (lfp_df["left_ma"] > 0)].copy()
spike_off_df = lfp_df[(lfp_df["prev_left"] > 0)  & (lfp_df["left_ma"] == 0)].copy()

spike_meta = []
for i, (_, row) in enumerate(spike_on_df.iterrows()):
    after  = spike_off_df[spike_off_df["eprime_ms"] > row["eprime_ms"]]
    off_ms = float(after.iloc[0]["eprime_ms"]) if len(after) else np.nan
    spike_meta.append({
        "spike_num"      : i + 1,
        "onset_ms"       : float(row["eprime_ms"]),
        "offset_ms"      : off_ms,
        "duration_ms"    : off_ms - float(row["eprime_ms"]) if not np.isnan(off_ms) else np.nan,
        "stim_ma"        : float(row["left_ma"]),
        "lfp_power_left" : float(row["left_lfp"]),
        "lfp_power_right": float(row["right_lfp"]),
    })
df_spike_meta = pd.DataFrame(spike_meta)
print(f"    Stimulation spikes : {len(df_spike_meta)}")

# ══════════════════════════════════════════════════════════════════════════════
# PART 3 — Build Event Marker Stream
# ══════════════════════════════════════════════════════════════════════════════
print("\n[PART 3] Building event marker stream...")

rows = []

def add(time_ms, code, trial=None, block=None, span=None,
        sub_trial=None, digit=None):
    rows.append({
        "Time_ms"          : float(time_ms),
        "Event_Code"       : int(code),
        "Trial_Number_raw" : trial,
        "Block"            : block,
        "Span_Size"        : span,
        "Sub_Trial"        : sub_trial,
        "Digit"            : digit,
    })

# Session Start — Welcome.TargetOnsetTime (same value every row, take first)
SESSION_START_MS = float(ep["session_start_ms"].iloc[0])
SESSION_END_MS   = float(ep["session_end_ms"].iloc[0])
add(SESSION_START_MS, 6)

trial_counter = 0
for (blk, trl), grp in ep.groupby(["block", "trial"]):
    trial_counter += 1
    grp   = grp.sort_values("sub_trial").reset_index(drop=True)
    span  = int(grp["span_size"].iloc[0])
    blk_i = int(blk)

    # Main Trial Start — first Fixation.TargetOnsetTime of trial
    add(grp["fixation_onset_ms"].min(), 8,
        trial=trial_counter, block=blk_i, span=span)

    # Per-digit events
    for _, row in grp.iterrows():
        sub = int(row["sub_trial"])
        dgt = int(row["digit_shown"])
        add(row["fixation_onset_ms"],  16, trial_counter, blk_i, span, sub, dgt)  # Fixation Start
        add(row["fixation_offset_ms"], 17, trial_counter, blk_i, span, sub, dgt)  # Fixation End
        add(row["stimulus_onset_ms"],  10, trial_counter, blk_i, span, sub, dgt)  # Stimulus Start
        add(row["stimulus_offset_ms"], 11, trial_counter, blk_i, span, sub, dgt)  # Stimulus End

    # Trial-level events — all come from last sub-trial row (E-Prime broadcasts
    # trial-level values to every sub-trial row; taking last is correct)
    last = grp.iloc[-1]
    add(last["response_onset_ms"],   12, trial_counter, blk_i, span)  # Choice Start
    add(last["response_offset_ms"],  13, trial_counter, blk_i, span)  # Choice End
    add(last["feedback_onset_ms"],   14, trial_counter, blk_i, span)  # Feedback Start
    add(last["feedback_offset_ms"],  15, trial_counter, blk_i, span)  # Feedback End

    # Main Trial End — last Feedback.OffsetTime of trial
    add(last["feedback_offset_ms"],   9, trial_counter, blk_i, span)

# Session End — Goodbye.OffsetTime
add(SESSION_END_MS, 7)

# Stimulation spike rows (removed from final CSV; kept in df_spike_meta)
for _, sp in df_spike_meta.iterrows():
    add(sp["onset_ms"],  18)
    if not np.isnan(sp["offset_ms"]):
        add(sp["offset_ms"], 19)

events_data = pd.DataFrame(rows).sort_values("Time_ms").reset_index(drop=True)
events_data["Time_s"] = events_data["Time_ms"] / 1000.0

n_spike_rows = len(events_data[events_data["Event_Code"].isin([18, 19])])
print(f"    Total event rows : {len(events_data)}  "
      f"(includes {n_spike_rows} spike rows)")

# ══════════════════════════════════════════════════════════════════════════════
# PART 4 — Event_Type labels
# ══════════════════════════════════════════════════════════════════════════════
print("\n[PART 4] Adding Event_Type labels...")
events_data["Event_Type"] = events_data["Event_Code"].map(EVENT_MAPPING)
print("    " + events_data["Event_Type"].value_counts().to_string().replace("\n", "\n    "))

# ══════════════════════════════════════════════════════════════════════════════
# PART 5 — Trial_Number
# ══════════════════════════════════════════════════════════════════════════════
print("\n[PART 5] Adding Trial_Number...")

def add_trial_numbers(df):
    out      = df.copy()
    out["Trial_Number"] = np.nan
    current  = 0
    in_trial = False
    for idx, row in out.iterrows():
        c = row["Event_Code"]
        if c == 8:
            current += 1; in_trial = True
            out.loc[idx, "Trial_Number"] = current
        elif c == 9:
            if in_trial:
                out.loc[idx, "Trial_Number"] = current
                in_trial = False
        else:
            if in_trial:
                out.loc[idx, "Trial_Number"] = current
    out["Trial_Number"] = out["Trial_Number"].astype("Int64")
    print(f"    Trials found         : {current}")
    print(f"    Events within trials : {out['Trial_Number'].notna().sum()}")
    print(f"    Events outside trials: {out['Trial_Number'].isna().sum()}")
    return out

events_data = add_trial_numbers(events_data)

# ══════════════════════════════════════════════════════════════════════════════
# PART 6 — Align to neural data
# ══════════════════════════════════════════════════════════════════════════════
print("\n[PART 6] Aligning events to neural data...")

def align_events_to_neural_data(df, sampling_rate, neural_start_ms, n_samples):
    """
    Time_Relative_s = (Time_ms - neural_start_ms) / 1000
    Sample_Index    = round(Time_Relative_s * sampling_rate)
      negative    → event before neural recording started
      0 to N-1    → event within neural recording window
      >= N_SAMPLES → event after neural recording ended
    """
    out = df.copy()
    out["Time_Relative_s"] = (out["Time_ms"] - neural_start_ms) / 1000.0
    out["Sample_Index"]    = (out["Time_Relative_s"] * sampling_rate).round().astype(int)

    n_end_ms = neural_start_ms + (n_samples - 1) / sampling_rate * 1000.0
    rec_dur  = n_samples / sampling_rate
    n_valid  = ((out["Sample_Index"] >= 0) & (out["Sample_Index"] < n_samples)).sum()
    n_before = (out["Sample_Index"] < 0).sum()
    n_after  = (out["Sample_Index"] >= n_samples).sum()

    print("    " + "=" * 56)
    print("    EVENT ALIGNMENT SUMMARY")
    print("    " + "=" * 56)
    print(f"    Neural idx 0       = E-Prime {neural_start_ms:.0f} ms ({neural_start_ms/1000:.3f} s)")
    print(f"    Neural samples     = {n_samples:,}  |  SR = {sampling_rate} Hz  |  dur = {rec_dur:.2f} s")
    print(f"    Neural idx {n_samples-1}  = E-Prime {n_end_ms:.0f} ms")
    print(f"    In recording window: {n_valid}")
    print(f"    Before recording   : {n_before}  (Sample_Index < 0)")
    print(f"    After recording    : {n_after}   (Sample_Index >= {n_samples})")
    if n_before:
        print(f"\n    ⚠  {n_before} events have negative Sample_Index")
        print(f"       → occurred before neural recording started")
        print(f"       → Time_ms and Time_Relative_s are still valid")
    print("    " + "=" * 56)
    return out

events_data = align_events_to_neural_data(events_data, SR, NEURAL_START_MS, N_SAMPLES)

# ══════════════════════════════════════════════════════════════════════════════
# PART 7 — Trial answers (ACC, CRESP, RESP)
# ══════════════════════════════════════════════════════════════════════════════
print("\n[PART 7] Adding trial answers...")

trial_answers = (
    ep.groupby(["block", "trial"])
    .agg(ACC=("is_correct",       "first"),
         CRESP=("correct_response","first"),
         RESP=("response_made",    "first"))
    .reset_index()
    .sort_values(["block", "trial"])
    .reset_index(drop=True)
)
trial_answers["Trial_Number"] = range(1, len(trial_answers) + 1)

def add_trial_answers(df, answers):
    out = df.copy()
    out["ACC"] = np.nan; out["CRESP"] = np.nan; out["RESP"] = np.nan
    for i, tn in enumerate(sorted(out["Trial_Number"].dropna().unique().tolist())):
        if i >= len(answers):
            print(f"    ⚠ No answer for trial {tn}"); continue
        a = answers.iloc[i]
        m = out["Trial_Number"] == tn
        out.loc[m, "ACC"]   = int(a["ACC"])
        out.loc[m, "CRESP"] = int(a["CRESP"])
        out.loc[m, "RESP"]  = int(a["RESP"])
    out["ACC"]   = out["ACC"].astype("Int64")
    out["CRESP"] = out["CRESP"].astype("Int64")
    out["RESP"]  = out["RESP"].astype("Int64")
    print(f"    Valid trials : {len(answers)}  |  "
          f"Correct : {(answers['ACC']==1).sum()}  |  "
          f"Incorrect : {(answers['ACC']==0).sum()}")
    return out

events_data = add_trial_answers(events_data, trial_answers)

# ══════════════════════════════════════════════════════════════════════════════
# PART 8 — Window labels (Stimulus / Choice / Feedback)
# ══════════════════════════════════════════════════════════════════════════════
print("\n[PART 8] Adding Window labels...")

def add_window_labels(df):
    """
    Stimulus window : Fixation Start (16) → last Stimulus End (11)
    Choice window   : Choice Start   (12) → Choice End (13)
    Feedback window : Feedback Start (14) → Feedback End (15)
    """
    out = df.copy()
    out["Window"] = pd.NA
    for tn in sorted(out["Trial_Number"].dropna().unique().tolist()):
        m  = out["Trial_Number"] == tn
        te = out[m]

        fs = te[te["Event_Code"] == 16]; se = te[te["Event_Code"] == 11]
        if len(fs) and len(se):
            out.loc[m & (out.index >= fs.index[0]) & (out.index <= se.index[-1]),
                    "Window"] = "Stimulus"

        cs = te[te["Event_Code"] == 12]; ce = te[te["Event_Code"] == 13]
        if len(cs) and len(ce):
            out.loc[m & (out.index >= cs.index[0]) & (out.index <= ce.index[0]),
                    "Window"] = "Choice"

        fbs = te[te["Event_Code"] == 14]; fbe = te[te["Event_Code"] == 15]
        if len(fbs) and len(fbe):
            out.loc[m & (out.index >= fbs.index[0]) & (out.index <= fbe.index[0]),
                    "Window"] = "Feedback"

    print("    " + "=" * 50)
    print("    WINDOW LABELING SUMMARY")
    print("    " + "=" * 50)
    for w in ["Stimulus", "Choice", "Feedback"]:
        sub = out[out["Window"] == w]
        if len(sub):
            print(f"      {w:10}: {len(sub)} events / {sub['Trial_Number'].nunique()} trials")
    print(f"      Unlabeled  : {out['Window'].isna().sum()} rows")
    print("    " + "=" * 50)
    return out

events_data = add_window_labels(events_data)

# ══════════════════════════════════════════════════════════════════════════════
# PART 9 — Final column ordering & save
# ══════════════════════════════════════════════════════════════════════════════
print("\n[PART 9] Finalising and saving...")

col_order = [
    "Time_ms", "Time_s",
    "Event_Code", "Event_Type",
    "Trial_Number", "Block", "Span_Size", "Sub_Trial", "Digit",
    "Window",
    "Time_Relative_s", "Sample_Index",
    "ACC", "CRESP", "RESP",
]
drop  = ["Trial_Number_raw"]
extra = [c for c in events_data.columns if c not in col_order and c not in drop]
events_data = events_data[[c for c in col_order if c in events_data.columns] + extra]

for col in ["Block", "Span_Size", "Sub_Trial", "Digit"]:
    events_data[col] = pd.array(events_data[col].values, dtype="Int64")

# Remove spike rows — post-task DBS cycling, not task-locked
# Full spike metadata retained in df_spike_meta for downstream use
events_data = events_data[~events_data["Event_Code"].isin([18, 19])].reset_index(drop=True)

events_data.to_csv(OUTPUT_PATH, index=False)
print(f"    ✓ Saved : {OUTPUT_PATH}")
print(f"    Shape   : {events_data.shape[0]} rows × {events_data.shape[1]} cols")

# ── Final summary ─────────────────────────────────────────────────────────────
print()
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"  Total rows       : {len(events_data)}")
print(f"  Unique trials    : {events_data['Trial_Number'].nunique()}")
n_in  = ((events_data["Sample_Index"] >= 0) & (events_data["Sample_Index"] < N_SAMPLES)).sum()
n_pre = (events_data["Sample_Index"] < 0).sum()
print(f"  In neural window : {n_in}")
print(f"  Pre-recording    : {n_pre}  (negative Sample_Index)")
print()
print("  Event type breakdown:")
for et, n in events_data["Event_Type"].value_counts().items():
    print(f"    {str(et):<26}: {n}")
print()
print("  Accuracy by span:")
ta = (events_data[events_data["Event_Code"] == 8]
      .groupby("Span_Size")["ACC"]
      .agg(["sum", "count"]))
for span, row in ta.iterrows():
    print(f"    {int(span)}-digit : {int(row['sum'])}/{int(row['count'])} = "
          f"{row['sum']/row['count']*100:.0f}%")
print()
print("Done.")

PIPELINE: E-Prime Preprocessor + Events Data Preprocessing

[PART 1] Building per-sub-trial table from E-Prime CSV...
    Raw shape : (52, 279)
    Sub-trials loaded : 52
    Blocks            : 7
    Session start     : 90205 ms  (Welcome.TargetOnsetTime)
    Session end       : 347047 ms   (Goodbye.OffsetTime)

[PART 2] Loading neural JSON...
    Neural SR       : 250.0 Hz
    Neural samples  : 70,000
    Neural window   : 348000 – 627996 ms (280.00 s)
    Stimulation spikes : 15

[PART 3] Building event marker stream...
    Total event rows : 324  (includes 30 spike rows)

[PART 4] Adding Event_Type labels...
    Event_Type
    Fixation Start           52
    Fixation End             52
    Stimulus Start           52
    Stimulus End             52
    Stimulation Spike        15
    Stimulation Spike End    15
    Main Trial Start         14
    Choice Start             14
    Choice End               14
    Feedback Start           14
    Feedback End             14
    Main Tria